# 0 · Testing MiniCPM as a chat model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MonicaCoraggioso/my-first-agents/blob/main/0-test_MiniCPM.ipynb)

A first experiment with a small open-source language model, **MiniCPM** (`openbmb/MiniCPM5-2B`), loaded from the Hugging Face Hub with 🤗 `transformers`.

**What this notebook does**
1. Loads the MiniCPM tokenizer and model.
2. Defines `generate_response()`, a small helper that sends a list of chat messages to the model and returns its reply.
3. Tries two prompts:
   - a simple question where the *system* message contains a fact the model should use;
   - a structured request (a function specification passed as JSON) to see if the model writes documented code.

**Requirements**
- Python packages: `torch`, `transformers`, `accelerate` (in Colab, only the last two need installing: `!pip install -q transformers accelerate`).
- About 5 GB of free disk space for the model weights, which download on first run.
- A GPU is recommended (in Colab: *Runtime → Change runtime type → T4 GPU*). On a Mac, PyTorch can use the Apple GPU (`mps`).

## 1. Imports

In [ ]:
import json                       # turn Python dicts into JSON strings for prompts
from typing import Dict, List     # type hints for the chat messages

import torch                      # deep-learning backend used by the model
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Load the model

`AutoTokenizer` and `AutoModelForCausalLM` download the files for `model_id` from the Hugging Face Hub the first time (they are cached afterwards).

- `torch_dtype="auto"` uses the precision the model was saved in (usually bfloat16), which halves memory compared with float32.
- `device_map="auto"` (provided by `accelerate`) puts the model on the GPU if one is available, otherwise on the CPU.

> Tip: the Hub may warn about *unauthenticated requests*. It still works; to silence it and get faster downloads, set a free `HF_TOKEN` (in Colab: the 🔑 *Secrets* panel).

In [ ]:
# Model to test: MiniCPM, a ~2-billion-parameter chat model from OpenBMB
model_id = "openbmb/MiniCPM5-2B"

# The tokenizer converts text <-> token IDs and knows the model's chat format
tokenizer = AutoTokenizer.from_pretrained(model_id)

# The model itself (weights are ~5 GB)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype="auto",   # keep the precision the weights were saved in
    device_map="auto",    # use the GPU automatically when available
)

print("Model loaded successfully!")

Model loaded successfully!


## 3. Helper: send a conversation, get a reply

`generate_response()` hides the three steps needed for every request:
1. **Format** – `apply_chat_template` turns the list of messages into the exact text format MiniCPM was trained on, and appends the cue for the assistant to start answering.
2. **Generate** – `model.generate` predicts new tokens one at a time.
3. **Decode** – only the newly generated tokens are turned back into text (the prompt is cut off).

In [ ]:
def generate_response(messages: List[Dict], max_new_tokens: int = 50) -> str:
    """Send a chat conversation to MiniCPM and return the model's reply.

    Args:
        messages: The conversation so far, as a list of dicts with a
            ``"role"`` ("system", "user" or "assistant") and a ``"content"``
            (the text). See the next cell for what each role means.
        max_new_tokens: Maximum number of tokens the model may generate.
            50 tokens is roughly 35-40 English words, so longer answers
            will be cut off mid-sentence; increase it for full answers.

    Returns:
        The model's reply as plain text (without the prompt).
    """
    # 1. Format the conversation with the model's chat template and convert it
    #    to token IDs (PyTorch tensors), on the same device as the model.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,   # add the "assistant's turn" marker
        enable_thinking=False,        # MiniCPM option: skip the hidden reasoning step
        return_dict=True,             # return input_ids and attention_mask
        return_tensors="pt",          # as PyTorch tensors
    ).to(model.device)

    # 2. Generate the answer. no_grad() turns off gradient tracking, which is
    #    only needed for training, so generation is faster and uses less memory.
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy decoding: same input -> same output
        )

    # 3. outputs[0] contains prompt + answer; keep only the new tokens
    #    (everything after the prompt length) and convert them back to text.
    prompt_length = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(
        outputs[0][prompt_length:],
        skip_special_tokens=True,     # drop markers like <|im_end|>
    )

    return response

### Chat message roles

Each message in `messages` has a **role**:

| Role | Purpose |
|---|---|
| `"system"` | Initial instructions, rules or context for how the model should behave for the whole session. Not part of the conversation itself, it sets the ground rules (e.g. *"You will respond in JSON."*). |
| `"user"` | Input from the user: prompts, questions or instructions. |
| `"assistant"` | Responses from the model. Include these to give the context of a conversation that has already started, or to guide the model with example answers. They are read as what the model said in the past. |

## 4. Example 1: a fact given in the system message

The system message sets a persona (*physics professor*) **and** states a fact (*the weather in London is cloudy*). The question checks whether the model uses that fact.

**Observation:** the answer mentions clouds but mostly gives a general description of London's climate rather than simply using the given fact, and it is cut off at 50 tokens.

In [ ]:
messages = [
    {"role": "system", "content": "You are my physics professor and the weather in London is cloudy."},
    {"role": "user", "content": "What's the weather like in London?"},
]

response = generate_response(messages)
print(response)

The weather in London can be quite unpredictable, especially with its temperate maritime climate. It's often cloudy, with occasional rain showers. If you're planning to go outside, it's a good idea to keep an umbrella handy. Also, the weather can


## 5. Example 2: structured input as JSON

Messages can carry any information as long as it is text. Here a function specification is stored as a Python dict and converted to a JSON string with `json.dumps`, so the model receives clear, structured instructions.

**Observation:** the model starts a well-documented `swap_keys_values` function, but the 50-token limit cuts it off before the code body. Call `generate_response(messages, max_new_tokens=300)` to get the complete function.

In [ ]:
# Specification of the function we want the model to write
code_spec = {
    "name": "swap_keys_values",
    "description": "Swaps the keys and values in a given dictionary.",
    "params": {
        "d": "A dictionary with unique values.",
    },
}

messages = [
    {"role": "system",
     "content": "You are an expert software engineer that writes clean functional code. You always document your functions."},
    # json.dumps turns the dict into a JSON string the model can read
    {"role": "user", "content": f"Please implement: {json.dumps(code_spec)}"},
]

response = generate_response(messages)
print(response)

```python
def swap_keys_values(d):
    """
    Swaps the keys and values in a given dictionary.

    Args:
        d (dict): A dictionary with unique values.

    Returns:
        dict: A new dictionary with
